In [150]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [151]:
df = pd.read_csv("global_air_quality_deforestation_dataset.csv")

df = (
    df.groupby(
        ["Country", "City", "Year"],
        as_index=False
    )
    .mean(numeric_only=True)
) #time series

In [152]:
def classify_aqi(aqi):
    if aqi < 50:
        return "Stable"
    elif aqi <= 150:
        return "Warning"
    else:
        return "Critical"

df["Risk_Level"] = df["AQI"].apply(
    classify_aqi
)

In [153]:
#convert location categories -> numerical values
country_encoder = LabelEncoder()

df["Country_Encoded"] = (
    country_encoder.fit_transform(
        df["Country"]
    )
)

In [154]:
#checking :))
print(df["Risk_Level"].value_counts())

Risk_Level
Warning     1151
Stable       573
Critical     526
Name: count, dtype: int64


In [155]:
print(df["Year"].min())
print(df["Year"].max())
print(df["Year"].nunique())

2014
2023
10


In [156]:
feature_cols = [
    "Country_Encoded",
    "Year",
    "PM2.5",
    "PM10",
    "Deforestation_Rate_%",
    "Afforestation_Rate_%",
    "Vehicles_Increase_%",
    "Industries_Increase_%",
    "Env_Budget_Million_USD",
    "Population_Density_Per_SqKm",
    "CO2_Emissions_MT",
    "Green_Space_Ratio_%",
    "Avg_Life_Expectancy_Index"
]

In [157]:
#normalize feature values
feature_scaler = MinMaxScaler()

X_scaled = feature_scaler.fit_transform(
    df[feature_cols]
)

#normalize AQI values
target_scaler = MinMaxScaler()

y_scaled = target_scaler.fit_transform(
    df[["AQI"]]
)

In [158]:
lookback = 5 #last 5 yrs (2014-2018)

X_sequences = []
y_sequences = []

for country in df["Country"].unique():

    country_df = (
        df[df["Country"] == country]
        .sort_values("Year")
        .reset_index(drop=True)
    )

    X_country = feature_scaler.transform(
        country_df[feature_cols]
    )

    y_country = target_scaler.transform(
        country_df[["AQI"]]
    )

    for i in range(
        len(country_df) - lookback
    ):

        X_sequences.append(
            X_country[
                i:i+lookback
            ]
        )

        y_sequences.append(
            y_country[
                i+lookback
            ]
        )

X_sequences = np.array(
    X_sequences
)

y_sequences = np.array(
    y_sequences
)

print(
    "X Shape:",
    X_sequences.shape
)

print(
    "y Shape:",
    y_sequences.shape
)

X Shape: (2025, 5, 13)
y Shape: (2025, 1)


In [159]:
#train test split
X_train, X_test, y_train, y_test = train_test_split(
    X_sequences,
    y_sequences,
    test_size=0.2,
    random_state=42,

)

In [160]:
#lstm
model = Sequential()

model.add(
    LSTM(
        64,
        return_sequences=True,
        input_shape=(
            X_train.shape[1],
            X_train.shape[2]
        )
    )
)

model.add(
    Dropout(0.2)
)

model.add(
    LSTM(
        32,
        return_sequences=False
    )
)

model.add(
    Dropout(0.2)
)

model.add(
    Dense(
        16,
        activation="relu"
    )
)

model.add(
    Dense(1)
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [161]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [162]:
#model training
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 0.0188 - mae: 0.0887 - val_loss: 0.0015 - val_mae: 0.0292
Epoch 2/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0055 - mae: 0.0540 - val_loss: 0.0018 - val_mae: 0.0333
Epoch 3/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0046 - mae: 0.0503 - val_loss: 0.0020 - val_mae: 0.0340
Epoch 4/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0038 - mae: 0.0450 - val_loss: 0.0017 - val_mae: 0.0312
Epoch 5/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0035 - mae: 0.0435 - val_loss: 0.0015 - val_mae: 0.0299
Epoch 6/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0032 - mae: 0.0415 - val_loss: 0.0014 - val_mae: 0.0285
Epoch 7/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0031 - mae: 0.0424 - val_loss: 0.0016 - val_mae: 0.0305
Epoch 8/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0029 - mae: 0.0404 - val_loss: 0.0015 - val_mae: 0.0297
Epoch 9/20
81/81 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0028 - mae:

In [163]:
loss, mae = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("MAE:", mae)

MAE: 0.02981969527900219


In [164]:
predictions = model.predict(
    X_test
)

predictions_real = (
    target_scaler.inverse_transform(
        predictions
    )
)

y_test_real = (
    target_scaler.inverse_transform(
        y_test
    )
)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


In [165]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)
import pandas as pd

predicted_labels = [
    classify_aqi(x[0])
    for x in predictions_real
]

actual_labels = [
    classify_aqi(x[0])
    for x in y_test_real
]

print("Predicted Counts:")
print(pd.Series(predicted_labels).value_counts())

print("\nActual Counts:")
print(pd.Series(actual_labels).value_counts())

# label order
labels = ["Stable", "Warning", "Critical"]

# confusion matrix
cm = confusion_matrix(
    actual_labels,
    predicted_labels,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{l}" for l in labels],
    columns=[f"Predicted_{l}" for l in labels]
)

print("\nConfusion Matrix:")
print(cm_df)

# Precision, Recall, F1 per class
print("\nClassification Report:")
print(classification_report(
    actual_labels,
    predicted_labels,
    labels=labels,
    target_names=labels,
    digits=4
))

# overall metrics
accuracy = accuracy_score(
    actual_labels,
    predicted_labels
)

precision = precision_score(
    actual_labels,
    predicted_labels,
    average='weighted'
)

recall = recall_score(
    actual_labels,
    predicted_labels,
    average='weighted'
)

f1 = f1_score(
    actual_labels,
    predicted_labels,
    average='weighted'
)

print("\nOverall Metrics:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

Predicted Counts:
Warning     220
Stable      105
Critical     80
Name: count, dtype: int64

Actual Counts:
Warning     212
Stable      102
Critical     91
Name: count, dtype: int64

Confusion Matrix:
                 Predicted_Stable  Predicted_Warning  Predicted_Critical
Actual_Stable                 102                  0                   0
Actual_Warning                  3                209                   0
Actual_Critical                 0                 11                  80

Classification Report:
              precision    recall  f1-score   support

      Stable     0.9714    1.0000    0.9855       102
     Warning     0.9500    0.9858    0.9676       212
    Critical     1.0000    0.8791    0.9357        91

    accuracy                         0.9654       405
   macro avg     0.9738    0.9550    0.9629       405
weighted avg     0.9666    0.9654    0.9649       405


Overall Metrics:
Accuracy:  0.9654
Precision: 0.9666
Recall:    0.9654
F1 Score:  0.9649


In [166]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(
    y_test_real,
    predictions_real
)

r2 = r2_score(
    y_test_real,
    predictions_real
)

print("AQI MAE:", mae)
print("AQI R²:", r2)

AQI MAE: 8.156354544219308
AQI R²: 0.9804441530272526


In [177]:
#forecasting - testing accuracy on 1 yr na nasa dataset
selected_country = "Philippines"
selected_city = "Davao"

city_df = (
    df[
        (df["Country"] == selected_country)
        & (df["City"] == selected_city)
    ]
    .sort_values("Year")
    .reset_index(drop=True)
)

# use 2014-2018 as input
test_data = city_df[
    city_df["Year"].between(2014, 2018)
]

# actual 2019 AQI
actual_2019 = city_df[
    city_df["Year"] == 2019
]["AQI"].values[0]

# scale input
sequence = feature_scaler.transform(
    test_data[feature_cols]
)

sequence = sequence.reshape(
    1,
    5,
    len(feature_cols)
)

# predict 2019
pred_scaled = model.predict(
    sequence,
    verbose=0
)

predicted_2019 = target_scaler.inverse_transform(
    pred_scaled
)[0][0]

print(f"Country: {selected_country}")
print(f"City: {selected_city}")
print(f"Actual 2019 AQI   : {actual_2019:.2f}")
print(f"Predicted 2019 AQI: {predicted_2019:.2f}")
print(f"Difference        : {abs(actual_2019 - predicted_2019):.2f}")

print(f"Actual Risk Level   : {classify_aqi(actual_2019)}")
print(f"Predicted Risk Level: {classify_aqi(predicted_2019)}")

Country: Philippines
City: Davao
Actual 2019 AQI   : 128.07
Predicted 2019 AQI: 141.74
Difference        : 13.67
Actual Risk Level   : Warning
Predicted Risk Level: Warning


In [181]:
#checking :))
print(f"Country: {selected_country}")
print(f"City: {selected_city}")
print()

print(city_df[["Year", "AQI"]].tail(10))

Country: Philippines
City: Davao

   Year         AQI
0  2014  148.088571
1  2015  127.922667
2  2016  143.816000
3  2017  139.481875
4  2018  114.621250
5  2019  128.070000
6  2020  129.675263
7  2021  139.907500
8  2022  132.123333
9  2023  118.753333


In [180]:
#next 5 yrs prediction after verification
selected_country = "Philippines"
selected_city = "Davao"

city_df = (
    df[
        (df["Country"] == selected_country)
        & (df["City"] == selected_city)
    ]
    .sort_values("Year")
    .reset_index(drop=True)
)

sequence_data = city_df[
    feature_cols
].tail(5)

current_sequence = feature_scaler.transform(
    sequence_data
)

current_sequence = current_sequence.reshape(
    1,
    5,
    len(feature_cols)
)

year_idx = feature_cols.index("Year")

future_predictions = []

for future_year in range(2024, 2029):

    pred_scaled = model.predict(
        current_sequence,
        verbose=0
    )

    pred_aqi = target_scaler.inverse_transform(
        pred_scaled
    )[0][0]

    future_predictions.append([
        future_year,
        round(pred_aqi, 2),
        classify_aqi(pred_aqi)
    ])

    # create next timestep
    next_row = current_sequence[0, -1].copy()

    # increment normalized year
    next_row[year_idx] += (
        1 /
        (
            df["Year"].max()
            -
            df["Year"].min()
        )
    )

    current_sequence = np.concatenate(
        [
            current_sequence[:, 1:, :],
            next_row.reshape(1, 1, -1)
        ],
        axis=1
    )

forecast_df = pd.DataFrame(
    future_predictions,
    columns=[
        "Year",
        "Predicted_AQI",
        "Risk_Level"
    ]
)

print(forecast_df)

   Year  Predicted_AQI Risk_Level
0  2024     134.250000    Warning
1  2025     132.160004    Warning
2  2026     131.679993    Warning
3  2027     127.800003    Warning
4  2028     125.589996    Warning


In [182]:
#save
import joblib

# save trained LSTM model
model.save("vergemap_lstm.keras")

# save scalers
joblib.dump(
    feature_scaler,
    "feature_scaler.pkl"
)

joblib.dump(
    target_scaler,
    "target_scaler.pkl"
)

# save country encoder
joblib.dump(
    country_encoder,
    "country_encoder.pkl"
)

print("All files saved successfully!")

All files saved successfully!


In [183]:
from google.colab import files

files.download("vergemap_lstm.keras")
files.download("feature_scaler.pkl")
files.download("target_scaler.pkl")
files.download("country_encoder.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>